In [0]:
tracks_path = "/Volumes/workspace/default/spotify_data/spotify_tracks.csv"
artists_path = "/Volumes/workspace/default/spotify_data/spotify_artists.csv"
albums_path = "/Volumes/workspace/default/spotify_data/spotify_albums.csv"

print(tracks_path)
print(artists_path)
print(albums_path)

In [0]:
tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(tracks_path)
)

In [0]:
tracks_df.printSchema()

In [0]:
print("Number of rows:", tracks_df.count())
print("Number of columns:", len(tracks_df.columns))

In [0]:
print(tracks_df.columns)

In [0]:
display(
    tracks_df.select(
        "id",
        "name",
        "album_id",
        "artists_id",
        "acousticness",
        "danceability",
        "energy",
        "loudness",
        "liveness",
        "speechiness",
        "tempo",
        "valence",
        "popularity",
        "duration_ms",
        "playlist"
    ).limit(10)
)

In [0]:
from pyspark.sql.functions import col, sum

null_counts = tracks_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in tracks_df.columns
])

display(null_counts)

In [0]:
from pyspark.sql.functions import col, sum, trim

string_columns = [
    field.name
    for field in tracks_df.schema.fields
    if field.dataType.simpleString() == "string"
]

empty_counts = tracks_df.select([
    sum(
        (trim(col(c)) == "").cast("int")
    ).alias(c)
    for c in string_columns
])

display(empty_counts)

In [0]:
from pyspark.sql.functions import count

duplicate_ids = (
    tracks_df
    .groupBy("id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

print("Duplicate track IDs:", duplicate_ids.count())

In [0]:
total_rows = tracks_df.count()
unique_rows = tracks_df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

In [0]:
from pyspark.sql.functions import col

print("artists_id = [] :", tracks_df.filter(col("artists_id") == "[]").count())
print("available_markets = [] :", tracks_df.filter(col("available_markets") == "[]").count())

In [0]:
numeric_features = [
    "acousticness",
    "danceability",
    "energy",
    "liveness",
    "valence",
    "speechiness",
    "tempo",
    "loudness",
    "duration_ms",
    "popularity"
]

display(
    tracks_df
    .select(numeric_features)
    .summary("count", "min", "mean", "max")
)

In [0]:
tempo_zero_count = tracks_df.filter(col("tempo") == 0).count()

print("Tracks with tempo = 0:", tempo_zero_count)

In [0]:
display(
    tracks_df
    .filter(col("tempo") == 0)
    .select(
        "id",
        "name",
        "duration_ms",
        "energy",
        "danceability",
        "speechiness",
        "popularity",
        "playlist"
    )
    .limit(20)
)

In [0]:
print("Tracks shorter than 30 seconds:",
      tracks_df.filter(col("duration_ms") < 30000).count())

In [0]:
short_and_zero_tempo = tracks_df.filter(
    (col("duration_ms") < 30000) &
    (col("tempo") == 0)
).count()

print("Tracks < 30s AND tempo = 0:", short_and_zero_tempo)

In [0]:
print(
    "Tracks longer than 20 minutes:",
    tracks_df.filter(col("duration_ms") > 1200000).count()
)

In [0]:
display(
    tracks_df
    .filter(col("duration_ms") > 1200000)
    .select(
        "id",
        "name",
        "duration_ms",
        "tempo",
        "energy",
        "danceability",
        "speechiness",
        "popularity",
        "playlist"
    )
    .orderBy(col("duration_ms").desc())
    .limit(20)
)

In [0]:
print(
    "Tracks with duration <= 0:",
    tracks_df.filter(col("duration_ms") <= 0).count()
)

In [0]:
display(
    tracks_df.select("popularity")
    .summary("count", "min", "mean", "max")
)

In [0]:
display(
    tracks_df.select("loudness")
    .summary("count", "min", "mean", "max")
)

In [0]:
positive_loudness_count = tracks_df.filter(
    col("loudness") > 0
).count()

print("Tracks with loudness > 0:", positive_loudness_count)

In [0]:
display(
    tracks_df
    .filter(col("loudness") > 0)
    .select(
        "id",
        "name",
        "loudness",
        "energy",
        "danceability",
        "tempo",
        "popularity",
        "playlist"
    )
    .orderBy(col("loudness").desc())
    .limit(20)
)

In [0]:
display(
    tracks_df.select(
        "key",
        "mode",
        "time_signature"
    ).summary("count", "min", "mean", "max")
)

In [0]:
display(
    tracks_df
    .groupBy("time_signature")
    .count()
    .orderBy("time_signature")
)

In [0]:
artists_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(artists_path)
)

print("Rows:", artists_df.count())
print("Columns:", len(artists_df.columns))

artists_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, count

# 1) NULL values
artist_nulls = artists_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in artists_df.columns
])

display(artist_nulls)

In [0]:
# 2) Duplicate artist IDs
artist_duplicate_ids = (
    artists_df
    .groupBy("id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

print("Duplicate artist IDs:", artist_duplicate_ids.count())

In [0]:
# 3) Empty genre lists
print(
    "Artists with genres = []:",
    artists_df.filter(col("genres") == "[]").count()
)

In [0]:
albums_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(albums_path)
)

print("Rows:", albums_df.count())
print("Columns:", len(albums_df.columns))

albums_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, count

album_nulls = albums_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in albums_df.columns
])

display(album_nulls)

In [0]:
album_duplicate_ids = (
    albums_df
    .groupBy("id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
)

print("Duplicate album IDs:", album_duplicate_ids.count())

In [0]:
display(
    albums_df
    .groupBy("album_type")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
album_nulls = albums_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in albums_df.columns
])
display(album_nulls)

In [0]:
album_duplicate_ids = (
    albums_df
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate album IDs:", album_duplicate_ids.count())

In [0]:
tracks_path = "/Volumes/workspace/default/spotify_data/spotify_tracks.csv"

tracks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(tracks_path)
)

print("Rows:", tracks_df.count())
print("Columns:", len(tracks_df.columns))

In [0]:
tracks_neo4j = tracks_df.select(
    "id",
    "name",
    "album_id",
    "artists_id",
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "popularity",
    "speechiness",
    "tempo",
    "time_signature",
    "valence",
    "playlist"
)

print("Rows:", tracks_neo4j.count())
print("Columns:", len(tracks_neo4j.columns))

In [0]:
tracks_neo4j = tracks_df.select(
    "id",
    "name",
    "album_id",
    "artists_id",
    "acousticness",
    "danceability",
    "duration_ms",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "popularity",
    "speechiness",
    "tempo",
    "time_signature",
    "valence",
    "playlist"
)

print("Rows:", tracks_neo4j.count())
print("Columns:", len(tracks_neo4j.columns))

In [0]:
output_dir = "/Volumes/workspace/default/spotify_data/neo4j_tracks_export"

(
    tracks_neo4j
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_dir)
)

display(dbutils.fs.ls(output_dir))

In [0]:
files = dbutils.fs.ls(output_dir)

part_file = [
    f.path for f in files
    if f.name.startswith("part-") and f.name.endswith(".csv")
][0]

destination = "/Volumes/workspace/default/spotify_data/spotify_tracks_neo4j.csv"

dbutils.fs.cp(part_file, destination)

print("Created:", destination)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/spotify_data"))